# Transform

In [139]:
import pandas as pd

import os,sys
import json
import argparse
import os.path
bic_etl_home = os.getenv('bic_etl_home')
## Add the bic_etl/general/script directory to path 
sys.path.insert(0, os.path.join(bic_etl_home, 'general', 'scripts'))
import custom_log
import jcLib
import gspread
from oauth2client.service_account import ServiceAccountCredentials


In [142]:
## Parse Input Arguments
parser = argparse.ArgumentParser(description='generic BIC transform script')
parser.add_argument('-t', '--title', help='-t is the dataset title')
parser.add_argument('-w', '--w4x4', help='-w is the CIM 4x4 identifier of the dataset')
args, leftovers = parser.parse_known_args()


#w4x4 = args.w4x4


## Logger
# logger = custom_log.setup(args.title)
# logger.info("Starting: Septic Systems in Boulder County Colorado")





In [143]:
def getInfo():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',
     scope)
    client = gspread.authorize(creds)

    # gc = gspread.service_account("/home/joe/work/client_secret.json")
    # for gg in gc.list_spreadsheet_files():
    #      print("GGGGG ",gg)
    
    tracker = client.open('cdos datasets for Python ETL transition').worksheet('datasets')
   
    df = pd.DataFrame(tracker.get_all_records(head=1))

    return  df
            
df=getInfo()

In [144]:
dataset="Paid Solicitors Disclosed on Charity Registration Forms in Colorado"
args.title=dataset
tmp=df.loc[df['title']==dataset]

In [145]:
group=tmp['group'].values.tolist()[0]
dictFile=tmp['dictionary'].values.tolist()[0]
inFile=tmp['infile'].values.tolist()[0]
w4x4=tmp['4x4'].values.tolist()[0]

In [146]:
logger2 = custom_log.setupNew(args.title)
logger2.info("Starting: Septic Systems in Boulder County Colorado",extra={"s4x4":w4x4})

{"name": "Paid Solicitors Disclosed on Charity Registration Forms in Colorado","4x4": "wwbh-7bpa", "pid": "18422", "level": "20", "msg": "Starting: Septic Systems in Boulder County Colorado", "time": "2024-06-07 20:55:34,399"}


In [148]:
dirTarget = f'{bic_etl_home}/{group}'
inFile=inFile.split(".")[0]+".tsv"
dictFile=open(f'{dirTarget}/defs/{dictFile}')
dataFile=open(f'{dirTarget}/data_source/{inFile}')
etlInfoFile=open(f'{dirTarget}/run_etl_new.json')


dictLookup = json.load(dictFile)
dictFile.close()
fieldXrefs={srcField:refs['xref'] for srcField,refs in dictLookup[w4x4].items()}

In [149]:

etlInfo=json.load(etlInfoFile)
etlInfoFile.close()

In [150]:
for etl in etlInfo:
    if etl['title'] == dataset :
        trans=etl['transform']
        outFile=trans['outputFile']
        inputFile=trans['inputFile']
        delimiter=trans['delimiter']
        print(trans)

{'language': 'node', 'delimiter': '\t', 'inputFile': 'data_source/char_orgs_sol.tsv', 'file': 'scripts/char_orgs_sol.js', 'outputFile': 'data_transformed/char_orgs_sol.tsv'}


In [151]:
df = pd.read_csv(f'{dirTarget}/data_source/{inFile}',delimiter=delimiter,encoding='latin')

for col in df.columns:
    if col not in fieldXrefs:
        print("Error")
        
df.rename(columns=fieldXrefs,inplace=True)
df.to_csv(f'{dirTarget}/{outFile}',index=False)

/home/joe/anaconda3/envs/bic/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3072: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)
